# 13. Debugging · Quality · Config

문제를 빠르게 찾고, 문서 품질을 자동 점검하고, App 기본 설정을 공유하는 도구.

이 튜토리얼의 모든 출력은 **실제 HWP 실행 결과**입니다 (hidden cell 이 HWP
subprocess 를 호출해 캡처).


In [ ]:
import sys
sys.path.insert(0, '.')
from tutorial_helpers import run_and_show

## 1. Discovery — `app.help()` / `repr(app)`

```python
app = App()
app.help()       # 18개 accessor + context manager 출력
repr(app)        # 상태 요약
```

실제 출력:


In [ ]:
from hwpapi.core import App
a = App(is_visible=False)
print(repr(a))
print()
a.help()
try: a.quit()
except Exception: pass

## 2. app.debug

### 2.1 state() — 현재 상태 dict

```python
state = app.debug.state()
```

실제 반환 값:


In [ ]:
from hwpapi.core import App
import pprint
a = App(is_visible=False)
try: a.api.Run("FileNew")
except Exception: pass
a.insert_text("샘플 텍스트")
a.set_charshape(bold=True, fontsize=1100)
state = a.debug.state()
pprint.pprint(state, width=70)
try: a.quit()
except Exception: pass

### 2.2 print() — 상태 예쁘게 출력

```python
app.debug.print()
```

실제 출력:


In [ ]:
from hwpapi.core import App
a = App(is_visible=False)
try: a.api.Run("FileNew")
except Exception: pass
a.insert_text("디버그 대상 문서 예시\n")
a.insert_text("두 번째 줄입니다")
a.debug.print()
try: a.quit()
except Exception: pass

### 2.3 timing(fn) — 함수 실행 시간 측정

```python
r = app.debug.timing(app.insert_text, "테스트 텍스트")
print(r)
```

실제 결과:


In [ ]:
from hwpapi.core import App
import pprint
a = App(is_visible=False)
try: a.api.Run("FileNew")
except Exception: pass

# 1) 텍스트 삽입 시간
r1 = a.debug.timing(a.insert_text, "측정 대상 텍스트")
print("insert_text('측정 대상 텍스트'):")
pprint.pprint(r1, width=70)

# 2) page_count property 접근 시간
r2 = a.debug.timing(lambda: a.page_count)
print("\npage_count:")
pprint.pprint(r2, width=70)

# 3) lint() 호출 시간
r3 = a.debug.timing(a.lint)
print("\nlint():")
print(f"  elapsed_ms: {r3['elapsed_ms']}")
print(f"  success: {r3['success']}")

try: a.quit()
except Exception: pass

### 2.4 trace() — COM Run() 호출 추적

```python
with app.debug.trace():
    app.preset.title_box(text="Test")
```

`#| echo: false` 로 hidden 된 블록의 실제 trace 출력:


In [ ]:
from hwpapi.core import App
a = App(is_visible=False)
try: a.api.Run("FileNew")
except Exception: pass
with a.debug.trace(verbose=True):
    a.insert_text("trace 대상")
    a.api.Run("SelectAll")
    a.api.Run("Cancel")
try: a.quit()
except Exception: pass

## 3. app.lint() — 문서 품질 체크

```python
report = app.lint()
print(report.summary)
```

실제 lint 결과 (샘플 문서 — 긴 문장 + 빈 문단 + 이중 공백 포함):


In [ ]:
from hwpapi.core import App
a = App(is_visible=False)
try: a.api.Run("FileNew")
except Exception: pass

# 품질 문제가 있는 샘플 문서 작성
a.insert_text("정상 문장입니다.")
a.api.Run("BreakPara")
a.insert_text("이 문장은 너무 길어서 린트 경고가 발생할 정도로 길게 작성되어야 하는데 정말 끝도 없이 이어지는 문장의 예시를 만들기 위해 계속 쓰고 있습니다.")
a.api.Run("BreakPara")
a.api.Run("BreakPara")   # 빈 문단
a.insert_text("이 문장에는  이중  공백이  있습니다.")
a.api.Run("BreakPara")
a.insert_text("마지막 정상 문장.")

report = a.lint()
print(report.summary)
print()
print("상세 — long_sentences:", report.long_sentences)
print("상세 — empty_paragraphs:", report.empty_paragraphs)
print("상세 — double_spaces:", report.double_spaces)
print("has_issues:", report.has_issues)
print("issue_count:", report.issue_count)

try: a.quit()
except Exception: pass

*위 report 가 감지한 문서:*

In [ ]:
run_and_show("lint_target", '''
app.set_charshape(bold=True, height=1600)
app.insert_text("품질 체크 대상 문서\n\n")
app.set_charshape(bold=False, height=1100)
app.insert_text("이 문장은 너무 길어서 린트 경고가 발생할 정도로 길게 작성되어야 하는데 정말 끝도 없이 이어지는 문장의 예시를 만들기 위해 계속 쓰고 있습니다.\n\n")
app.insert_text("\n\n")
app.insert_text("이 문장에는  이중  공백이  있습니다.\n\n")
app.insert_text("정상적인 길이의 문장.\n")
''')

## 4. app.template — 문서 템플릿

```python
app.template.save("company_style.json")
```

저장된 JSON 실제 내용:


In [ ]:
import json, tempfile, os
from hwpapi.core import App
a = App(is_visible=False)
try: a.api.Run("FileNew")
except Exception: pass

# 세팅
a.insert_text("템플릿 소스 문서")
a.set_charshape(bold=True, fontsize=1100, text_color="#000000")

# 저장
tmp = tempfile.NamedTemporaryFile(suffix='.json', delete=False).name
data = a.template.save(tmp)

# 저장된 파일 읽기
with open(tmp, encoding='utf-8') as f:
    saved = json.load(f)
print(json.dumps(saved, ensure_ascii=False, indent=2))

os.remove(tmp)
try: a.quit()
except Exception: pass

### template.apply — 다른 문서에 적용

```python
app2 = App()
app2.open("new_doc.hwp")
app2.template.apply("company_style.json")
# → charshape, parashape 일괄 반영
```


## 5. app.config — App 기본 선호도

```python
app.config.default_font = "함초롬바탕"
app.config.default_size = 11
app.config.palette = {
    "primary": "#003366",
    "accent": "#FF6600",
}
app.config.to_dict()
```

실제 출력:


In [ ]:
import pprint
from hwpapi.core import App
a = App(is_visible=False)

# 설정
a.config.default_font = "함초롬바탕"
a.config.default_size = 11
a.config.default_line_spacing = 160
a.config.palette = {
    "primary": "#003366",
    "accent": "#FF6600",
    "bg_alt": "#EEEEEE",
}

# dict 로
print(">>> app.config.to_dict()")
pprint.pprint(a.config.to_dict(), width=70)

# reset 동작
print("\n>>> app.config.reset()")
a.config.reset()
pprint.pprint(a.config.to_dict(), width=70)

try: a.quit()
except Exception: pass

### config 디스크 저장 / 로드

```python
app.config.save("~/.hwpapirc")       # 저장
app2.config.load("~/.hwpapirc")      # 다른 세션에서 로드
```

설정 파일 (JSON):


In [ ]:
import json, tempfile, os
from hwpapi.core import App
a = App(is_visible=False)
a.config.update(
    default_font="함초롬바탕",
    default_size=11,
    palette={"primary": "#003366"},
)
tmp = tempfile.NamedTemporaryFile(suffix='.json', delete=False).name
a.config.save(tmp)
with open(tmp, encoding='utf-8') as f:
    print(f.read())
os.remove(tmp)
try: a.quit()
except Exception: pass

## 6. 실전 디버그 세션

```python
app = App()
app.open("problem.hwp")

# 1. 상태 파악
app.debug.print()

# 2. 문서 품질 체크
report = app.lint()
if report.has_issues:
    print(report.summary)

# 3. 느린 작업 조사
for fn_name in ["current_page", "page_count", "lint"]:
    fn = getattr(app, fn_name)
    r = app.debug.timing(lambda: fn() if callable(fn) else fn)
    print(f"{fn_name:20s} {r['elapsed_ms']:.1f}ms")

# 4. 특정 작업 trace
with app.debug.trace():
    app.preset.striped_rows()
```

위 세션을 실제로 실행한 결과:


In [ ]:
from hwpapi.core import App
a = App(is_visible=False)
try: a.api.Run("FileNew")
except Exception: pass

# 샘플 문서 작성
a.insert_text("실전 디버그 세션 샘플 문서입니다.\n")
a.insert_text("두 번째 문단.\n")
a.insert_text("이 문장은 너무 길어서 린트 경고를 트리거할 수 있도록 길게 작성한 예시입니다.\n")

# 1. 상태
print("=" * 50)
print(" [1] 상태 파악")
print("=" * 50)
a.debug.print()

# 2. 품질
print("\n" + "=" * 50)
print(" [2] 문서 품질 체크")
print("=" * 50)
r = a.lint()
print(r.summary)

# 3. 성능
print("\n" + "=" * 50)
print(" [3] 느린 작업 조사")
print("=" * 50)
for fn_name in ["current_page", "page_count", "lint"]:
    fn = getattr(a, fn_name)
    timing = a.debug.timing(lambda fn=fn: fn() if callable(fn) else fn)
    print(f"  {fn_name:20s} {timing['elapsed_ms']:6.2f}ms")

try: a.quit()
except Exception: pass